In [25]:
from gurobipy import * 

In [26]:
facility, max_cap, fixed_cost = multidict({
                      'SanFran': [1700, 70000],
                      'LA': [2000, 70000],
                      'Pheonix': [1700, 65000],
                      'Denver': [2000, 70000]
})

In [27]:
supply_store, max_dem = multidict({
                        'SD': 1700,
                        'Barstow': 1000, 
                        'Tucson': 1500, 
                        'Dallas': 1200
})

In [28]:
shipment_costs = {
                  ('SanFran', 'SD'): 5, 
                  ('SanFran', 'Barstow'): 3, 
                  ('SanFran', 'Tucson'): 2, 
                  ('SanFran', 'Dallas'): 6, 
                  ('LA', 'SD'): 4, 
                  ('LA', 'Barstow'): 7, 
                  ('LA', 'Tucson'): 8, 
                  ('LA', 'Dallas'): 10, 
                  ('Pheonix', 'SD'): 6, 
                  ('Pheonix', 'Barstow'): 5, 
                  ('Pheonix', 'Tucson'): 3, 
                  ('Pheonix', 'Dallas'): 8, 
                  ('Denver', 'SD'): 9, 
                  ('Denver', 'Barstow'): 8, 
                  ('Denver', 'Tucson'): 6, 
                  ('Denver', 'Dallas'): 5 
                                    }

In [29]:
facloc = Model("Facility_location_example")

Set parameter Username
Set parameter LicenseID to value 2625267
Academic license - for non-commercial use only - expires 2026-02-19


In [30]:
y = facloc.addVars(facility, vtype = GRB.BINARY, name= "facilities")

#Defining decision variables for demand and capacity at the warehouses and plants respectively
demand = (len(facility))
capacity = range(len(supply_store))
print (demand)
print(capacity)

In [31]:
x = facloc.addVars(facility, supply_store, lb = 0, vtype = GRB.CONTINUOUS, name="ship_amnt")

In [32]:
max_cap_constr = facloc.addConstrs((x.sum(f, '*') <= max_cap[f] for f in facility),name="capacity_constr")

In [33]:
max_dem_constr = facloc.addConstrs((x.sum('*', s) == max_dem[s] for s in supply_store),name="demand_constr")

In [34]:
obj = sum(x[f,s]*shipment_costs[f,s] for f in facility for s in supply_store)

In [35]:
facloc.setObjective(obj, GRB.MINIMIZE)

facloc.optimize()

Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 23.5.0 23F79)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 8 rows, 20 columns and 36 nonzeros
Model fingerprint: 0x16f2dfe1
Variable types: 16 continuous, 4 integer (4 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [2e+00, 7e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+03, 2e+03]
Presolve time: 0.00s
Presolved: 8 rows, 20 columns, 36 nonzeros
Variable types: 16 continuous, 4 integer (4 binary)
Found heuristic solution: objective 294600.00000

Root relaxation: objective 2.188294e+05, 9 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 218829.412    0    1 294600.000 218829.412  25.7%     -    0s
H    0     0                    230300.0

In [36]:
for v in facloc.getVars():
    if v.X !=0:
        print (v.VarName, v.X)

facilities[SanFran] 1.0
facilities[LA] 1.0
facilities[Pheonix] 1.0
ship_amnt[SanFran,Barstow] 700.0
ship_amnt[SanFran,Dallas] 1000.0
ship_amnt[LA,SD] 1700.0
ship_amnt[LA,Barstow] 300.0
ship_amnt[Pheonix,Tucson] 1500.0
ship_amnt[Pheonix,Dallas] 200.0


In [ ]:
for f in facility:
    for s in supply_store:
            if x[f, s].X > 0:
                print(f"  Transport {x[f,s].X:g} units from {f} to supply store {s}")